In [1]:
!pip install -q requests beautifulsoup4 lxml
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
import re
import time
import csv
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
BASE_URL   = "https://www.federalreserve.gov"
OUTPUT_CSV = "fomc_raw.csv"
START_DATE = "2016-01-01"
END_DATE   = "2025-12-31"
SLEEP_SEC  = 1.5

HEADERS = {"User-Agent": "Mozilla/5.0 (academic research; requests/beautifulsoup4)"}

print(f"Config loaded  |  Range: {START_DATE} → {END_DATE}  |  Output: {OUTPUT_CSV}")

Config loaded  |  Range: 2016-01-01 → 2025-12-31  |  Output: fomc_raw.csv


In [3]:
def get(url):
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml")

def parse_date(url):
    m = re.search(r"(\d{8})", url)
    if m:
        d = m.group(1)
        return f"{d[:4]}-{d[4:6]}-{d[6:]}"
    return None

def in_range(date):
    return START_DATE <= date <= END_DATE

def full_url(href):
    return href if href.startswith("http") else BASE_URL + href

def is_minutes(href):
    return bool(re.search(r"/monetarypolicy/fomcminutes\d{8}\.htm", href, re.IGNORECASE))

def is_statement(href):
    return bool(re.search(r"/newsevents/pressreleases/monetary\d{8}a\.htm", href, re.IGNORECASE))

def scrape_text(url):
    soup = get(url)
    body = (
        soup.find("div", id="content")
        or soup.find("article")
        or soup.find("main")
        or soup.body
    )
    if body:
        for tag in body.find_all(["nav", "header", "footer", "script", "style", "aside"]):
            tag.decompose()
        return body.get_text(separator=" ", strip=True)
    return ""

print("✅ Helper functions defined")

✅ Helper functions defined


In [4]:
def collect_links():
    links = []
    years = range(int(START_DATE[:4]), int(END_DATE[:4]) + 1)

    for year in years:
        page_url = (
            f"{BASE_URL}/monetarypolicy/fomchistorical{year}.htm"
            if year <= 2019
            else f"{BASE_URL}/monetarypolicy/fomccalendars.htm"
        )
        print(f"🔍  Scanning {year}: {page_url}")
        try:
            soup = get(page_url)
        except Exception as e:
            print(f"   ⚠️  Failed: {e}")
            continue

        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            url  = full_url(href)
            date = parse_date(url)

            if not date or not in_range(date):
                continue

            if is_minutes(href):
                links.append({"date": date, "doc_type": "minutes", "url": url})
            elif is_statement(href):
                links.append({"date": date, "doc_type": "statement", "url": url})

        time.sleep(SLEEP_SEC)

    # Deduplicate by (date, doc_type)
    seen, unique = set(), []
    for item in links:
        key = (item["date"], item["doc_type"])
        if key not in seen:
            seen.add(key)
            unique.append(item)

    return sorted(unique, key=lambda x: (x["date"], x["doc_type"]))


links = collect_links()

mins  = sum(1 for l in links if l["doc_type"] == "minutes")
stmts = sum(1 for l in links if l["doc_type"] == "statement")
print(f"\n📋  Found: {mins} minutes  |  {stmts} statements  |  {len(links)} total")

🔍  Scanning 2016: https://www.federalreserve.gov/monetarypolicy/fomchistorical2016.htm
🔍  Scanning 2017: https://www.federalreserve.gov/monetarypolicy/fomchistorical2017.htm
🔍  Scanning 2018: https://www.federalreserve.gov/monetarypolicy/fomchistorical2018.htm
🔍  Scanning 2019: https://www.federalreserve.gov/monetarypolicy/fomchistorical2019.htm
🔍  Scanning 2020: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm
🔍  Scanning 2021: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm
🔍  Scanning 2022: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm
🔍  Scanning 2023: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm
🔍  Scanning 2024: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm
🔍  Scanning 2025: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm

📋  Found: 72 minutes  |  74 statements  |  146 total


In [5]:
rows   = []
errors = []

for i, item in enumerate(links, 1):
    label = f"[{i:>3}/{len(links)}]  {item['date']}  [{item['doc_type']:<10}]"
    try:
        text = scrape_text(item["url"])
        rows.append({
            "meeting_date": item["date"],
            "doc_type"    : item["doc_type"],
            "url"         : item["url"],
            "text"        : text
        })
        print(f"{label}  ✅  {len(text):,} chars")
    except Exception as e:
        errors.append({"item": item, "error": str(e)})
        print(f"{label}  ❌  {e}")

    time.sleep(SLEEP_SEC)

print(f"\n✅  Scraped: {len(rows)} docs  |  ❌  Errors: {len(errors)}")

[  1/146]  2016-01-27  [minutes   ]  ✅  90,720 chars
[  2/146]  2016-01-27  [statement ]  ✅  4,068 chars
[  3/146]  2016-03-16  [minutes   ]  ✅  51,609 chars
[  4/146]  2016-03-16  [statement ]  ✅  4,126 chars
[  5/146]  2016-04-27  [minutes   ]  ✅  61,876 chars
[  6/146]  2016-04-27  [statement ]  ✅  4,152 chars
[  7/146]  2016-06-15  [minutes   ]  ✅  54,840 chars
[  8/146]  2016-06-15  [statement ]  ✅  3,922 chars
[  9/146]  2016-07-27  [minutes   ]  ✅  68,193 chars
[ 10/146]  2016-07-27  [statement ]  ✅  4,066 chars
[ 11/146]  2016-09-21  [minutes   ]  ✅  71,873 chars
[ 12/146]  2016-09-21  [statement ]  ✅  4,298 chars
[ 13/146]  2016-11-02  [minutes   ]  ✅  61,549 chars
[ 14/146]  2016-11-02  [statement ]  ✅  4,270 chars
[ 15/146]  2016-12-14  [minutes   ]  ✅  55,845 chars
[ 16/146]  2016-12-14  [statement ]  ✅  3,929 chars
[ 17/146]  2017-02-01  [minutes   ]  ✅  83,326 chars
[ 18/146]  2017-02-01  [statement ]  ✅  3,714 chars
[ 19/146]  2017-03-15  [minutes   ]  ✅  59,855 chars
[ 

In [6]:
df = pd.DataFrame(rows, columns=["meeting_date", "doc_type", "url", "text"])
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"✅  {len(df)} rows saved → {OUTPUT_CSV}")
print(f"   Date range: {df['meeting_date'].min()} → {df['meeting_date'].max()}")
df.head()

✅  146 rows saved → fomc_raw.csv
   Date range: 2016-01-27 → 2025-12-10


,meeting_date,doc_type,url,text
0,2016-01-27,minutes,https://www.federalreserve.gov/monetarypolicy/...,Home Monetary Policy Federal Open Market Commi...
1,2016-01-27,statement,https://www.federalreserve.gov/newsevents/pres...,Home News & Events Press Releases Press Releas...
2,2016-03-16,minutes,https://www.federalreserve.gov/monetarypolicy/...,Home Monetary Policy Federal Open Market Commi...
3,2016-03-16,statement,https://www.federalreserve.gov/newsevents/pres...,Home News & Events Press Releases Press Releas...
4,2016-04-27,minutes,https://www.federalreserve.gov/monetarypolicy/...,Home Monetary Policy Federal Open Market Commi...


In [7]:
from google.colab import files
files.download("fomc_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>